# [Kaggle] PatchCore Training - All Categories

Builds PatchCore memory banks for MVTec AD using a Kaggle GPU.
Uses pretrained ResNet features (no gradient optimization required).

### Kaggle Setup Instructions:
1. Requires **Internet: ON** (to download ResNet weights!).
2. Requires **GPU: ON** (feature extraction is much faster on GPU).
3. Add the **MVTec AD** dataset.
4. Upload your project source code (`src/` folder).

In [ ]:
# ==============================================================================
# UNIFIED KAGGLE SETUP & IMPORTS
# ==============================================================================
import os
import sys
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import time
from tqdm import tqdm
from pathlib import Path
from sklearn.metrics import roc_auc_score, roc_curve

# 1. Exact paths we discovered earlier
KAGGLE_WORKING = '/kaggle/working'
MVTEC_PATH = '/kaggle/input/datasets/ipythonx/mvtec-ad'
SRC_PREFIX = '/kaggle/input/datasets/mohammadhameem/thesis-project-source-code'

# 2. Add source to path so we can import from 'src'
sys.path.insert(0, SRC_PREFIX)

# 3. Standard MVTec categories
MVTEC_CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid', 
    'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 
    'tile', 'toothbrush', 'transistor', 'wood', 'zipper'
]

# 4. Import our local project files & patch config
try:
    from src import config as proj_cfg
    proj_cfg.DATA_DIR = Path(MVTEC_PATH).parent
    proj_cfg.MVTEC_DIR = Path(MVTEC_PATH)
    
    from src.data import create_mvtec_dataloaders
    from src.models.patchcore import create_patchcore
    print("\u2705 MVTec paths and source code loaded successfully!")
except ImportError as e:
    print(f"\u274c ERROR IMPORTING SOURCE CODE: {e}")

# 5. Device Setup
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\u2705 Using device: {DEVICE}")

# 6. Create Output Dirs
MODELS_DIR = Path(KAGGLE_WORKING) / 'models'
FIGURES_DIR = Path(KAGGLE_WORKING) / 'figures'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Visualization Helpers

In [ ]:
def save_and_show(fig, path):
    fig.savefig(path, dpi=150, bbox_inches='tight')
    # Note: On Kaggle we usually want to draw the plot so it appears in the notebook output cell
    plt.show() 
    plt.close(fig)
    print(f"  Saved: {path}")

def plot_roc_curve(all_scores, all_labels, auc_val, category, save_path):
    fpr, tpr, _ = roc_curve(all_labels, all_scores)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'PatchCore (AUC = {auc_val:.4f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve \u2014 {category}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    save_and_show(fig, save_path)

## Build Memory Bank Function

In [ ]:
def build_patchcore_category(category):
    print(f"\n{'='*60}")
    print(f"PatchCore: {category.upper()}")
    print(f"{'='*60}")

    try:
        train_loader, test_loader = create_mvtec_dataloaders(
            category, batch_size=CONFIG['batch_size'], return_mask=True
        )
    except Exception as e:
        print(f"Skipping {category}: Data loading failed. Error: {e}")
        return None

    model = create_patchcore(
        backbone=CONFIG['backbone'],
        k=CONFIG['k'],
        subsample_ratio=CONFIG['subsample_ratio']
    )

    start_time = time.time()
    print("  Extracting features and building memory bank...")
    model.fit(train_loader, device=DEVICE)
    fit_time = time.time() - start_time
    print(f"  Built in {fit_time:.1f}s | Size: {model.memory_bank.shape[0]}")

    print("  Evaluating on test set...")
    model.eval()
    all_scores, all_labels = [], []
    
    with torch.no_grad():
        # Using tqdm for evaluation progress
        for img, mask, label in test_loader:
            img = img.to(DEVICE)
            scores = model.get_anomaly_score(img)
            all_scores.extend(scores.cpu().numpy())
            all_labels.extend(label.numpy())

    try:
        auc = roc_auc_score(all_labels, all_scores)
        print(f"  >>> {category.upper()} ROC-AUC: {auc:.4f} <<<")
    except:
        auc = 0.0

    # Save model
    save_path = MODELS_DIR / f'patchcore_{category}_memory.pth'
    model.save_memory_bank(str(save_path))
    print(f"  Saved memory bank.")

    # Plots
    if auc > 0:
        plot_roc_curve(all_scores, all_labels, auc, category, FIGURES_DIR / f'patchcore_{category}_roc.png')

    return {
        'category': category,
        'auc': auc,
        'fit_time_s': round(fit_time, 1),
        'memory_bank_size': model.memory_bank.shape[0]
    }

## Execute Run

In [ ]:
results = []
total_start = time.time()

# 1. Define configuration right here to ensure it's always loaded!
CONFIG = {
    'batch_size': 16,        # Increased for GPU
    'backbone': 'resnet18',  # Must have internet ON to download!
    'k': 3,
    'subsample_ratio': 0.1,  # 10% subset (tradeoff between speed and accuracy)
}
CATEGORIES_TO_TRAIN = MVTEC_CATEGORIES
print(f"Will build {len(CATEGORIES_TO_TRAIN)} categories.")

# 2. Run the loop
for cat in CATEGORIES_TO_TRAIN:
    # Important on Kaggle: Clear cache to prevent OOM errors across categories
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
        
    res = build_patchcore_category(cat)
    if res:
        results.append(res)
        
total_elapsed = time.time() - total_start
print(f"\n{'='*60}")
print(f"ALL CATEGORIES COMPLETE in {total_elapsed/60:.1f} minutes")
print(f"{'='*60}")


## Summary Report

In [ ]:
if results:
    df = pd.DataFrame(results)
    # Save CSV
    csv_path = Path(KAGGLE_WORKING) / 'patchcore_results_all.csv'
    df.to_csv(csv_path, index=False)
    
    print("\n" + "="*40)
    print("PATCHCORE RESULTS SUMMARY")
    print("="*40)
    print(df.sort_values('auc', ascending=False).to_string(index=False))
    print(f"\nMean AUC: {df['auc'].mean():.4f}")
else:
    print("No successful runs.")

In [ ]:
from IPython.display import Image, display
from pathlib import Path

# Point to your figures directory
fig_dir = Path('/kaggle/working/figures')

# Find all PNG files and sort them alphabetically
image_files = sorted(fig_dir.glob('*.png'))

if not image_files:
    print("No images found in /kaggle/working/figures")
else:
    print(f"Found {len(image_files)} images. Displaying now...\n")
    for img_path in image_files:
        print(f"{'='*40}\n\U0001f5bc\ufe0f {img_path.name}\n{'='*40}")
        display(Image(filename=str(img_path)))
        print("\n") # Add a little breathing room between images